**Installing Required Libraries for Multimodal AI, Image Processing, and User Interface**

In [1]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q gradio deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.7 MB/s eta 0:00:00


**Importing Core Libraries for AI Model, Image Handling, and Interface Development**

In [2]:
import gradio as gr
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from PIL import Image
from deep_translator import GoogleTranslator

**Loading and Optimizing the LLaVA Multimodal Model for Image Understanding**

In [3]:
model_id = "llava-hf/llava-1.5-7b-hf"

print("Loading model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)

print("Model loaded successfully!")

Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

Model loaded successfully!


**Creating Translation Functions for Bengali Language Support**

In [4]:
from deep_translator import GoogleTranslator

def bn_to_en(text):
    try:
        return GoogleTranslator(source='bn', target='en').translate(text)
    except:
        return text

def en_to_bn(text):
    try:
        return GoogleTranslator(source='en', target='bn').translate(text)
    except:
        return text

In [5]:
def improve_bangla(text):
    text = text.replace("সাইকেল", "রিকশা")
    text = text.replace("কাঠি", "")
    text = text.replace("অলংকারিক", "সুন্দরভাবে সাজানো")
    text = text.replace("কার্ট", "গাড়ির অংশ")
    text = text.replace("ব্যক্তি", "একজন মানুষ")

    if not text.endswith("।"):
        text = text.rstrip("., ") + "।"

    if not text.startswith("ছবিতে"):
        text = "ছবিতে " + text

    return text

**Designing the Core Image Understanding Function Using LLaVA**

In [6]:
import torch
import re

def ask_ai(image, question_bn):
    try:
        print("Processing request...")

        if image is None:
            return "দয়া করে একটি ছবি আপলোড করুন।"

        if question_bn is None or question_bn.strip() == "":
            return "দয়া করে একটি প্রশ্ন লিখুন।"

        try:
            image = image.resize((224, 224))
        except Exception:
            return "ছবিটি সঠিকভাবে লোড হয়নি।"

        try:
            question_en = bn_to_en(question_bn)
        except Exception:
            return "প্রশ্ন অনুবাদ করতে সমস্যা হয়েছে।"

        prompt = f"""
You are an AI assistant.

Describe the image in ONE short, clear sentence.
Do not explain more than one sentence.

Focus on main objects like people, vehicles, street.

USER: <image>
{question_en}
ASSISTANT:
"""

        device = "cuda" if torch.cuda.is_available() else "cpu"

        try:
            inputs = processor(
                text=prompt,
                images=image,
                return_tensors="pt"
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
        except Exception as e:
            print("Processor error:", e)
            return "ইমেজ প্রসেস করতে সমস্যা হয়েছে।"

        try:
            with torch.no_grad():
                output = model.generate(
                    **inputs,
                    max_new_tokens=100,
                    do_sample=False,
                    eos_token_id=processor.tokenizer.eos_token_id
                )
        except Exception as e:
            print("Model error:", e)
            return "মডেল থেকে উত্তর তৈরি করতে সমস্যা হয়েছে।"

        answer = processor.decode(output[0], skip_special_tokens=True)

        if "ASSISTANT:" in answer:
            answer = answer.split("ASSISTANT:")[-1].strip()

        answer_en = answer.split("\n")[0]

        sentences = re.split(r'[.!?]', answer_en)
        if len(sentences) > 0 and sentences[0].strip():
            answer_en = sentences[0].strip() + "."
        else:
            answer_en = "Unable to describe the image."

        print("English:", answer_en)

        try:
            answer_bn = en_to_bn(answer_en)
        except Exception:
            return "বাংলায় অনুবাদ করতে সমস্যা হয়েছে।"

        if answer_bn.endswith("বা") or answer_bn.endswith("এবং"):
            answer_bn = answer_bn[:-2] + "।"

        try:
            answer_bn = improve_bangla(answer_bn)
        except:
            pass

        print("Bangla:", answer_bn)

        return answer_bn

    except Exception as e:
        print("ERROR:", e)
        return "অপ্রত্যাশিত সমস্যা হয়েছে।"

**Building a Simple and Interactive Gradio User Interface**

In [8]:
import gradio as gr

gr.close_all()

custom_css = """
body {
    font-family: 'Inter', sans-serif;
    background: #0b1220;
    color: #e5e7eb;
}

.gradio-container {
    max-width: 900px !important;
    margin: auto;
}

h1 {
    text-align: center;
    color: #38bdf8;
}

.section {
    background: #111827;
    padding: 18px;
    border-radius: 12px;
    margin-bottom: 16px;
}

button {
    border-radius: 8px !important;
    font-weight: 500 !important;
}

.footer {
    text-align: center;
    font-size: 12px;
    color: #9ca3af;
    margin-top: 20px;
}
"""

def validated_ask(image, question):
    if image is None:
        return "দয়া করে একটি ছবি আপলোড করুন।"
    if question is None or question.strip() == "":
        return "দয়া করে একটি প্রশ্ন লিখুন।"
    return ask_ai(image, question)


with gr.Blocks(css=custom_css) as demo:

    gr.Markdown("# BanglaVision AI")

    with gr.Group(elem_classes="section"):
        with gr.Row():
            with gr.Column():
                image_input = gr.Image(
                    type="pil",
                    label="Upload Image",
                    height=240
                )

            with gr.Column():
                question_input = gr.Textbox(
                    label="Your Question (Bangla)",
                    placeholder="Example: এই ছবিতে কি আছে?",
                    lines=4
                )

    with gr.Row():
        submit_btn = gr.Button("Submit")
        clear_btn = gr.Button("Clear")

    with gr.Group(elem_classes="section"):
        output = gr.Textbox(
            label="AI Response",
            lines=6
        )

    submit_btn.click(
        fn=validated_ask,
        inputs=[image_input, question_input],
        outputs=output,
        show_progress="full"
    )

    clear_btn.click(
        fn=lambda: (None, "", ""),
        inputs=[],
        outputs=[image_input, question_input, output]
    )

    gr.Markdown('<div class="footer">BanglaVision AI</div>')

demo.launch(share=True)

Closing server running on port: 7860


/tmp/ipykernel_4631/1986114437.py:50: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://831c8578279d4c03e9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
